# Reranker output analysis

Notebook này trực quan hóa và thống kê kết quả Jina reranker từ file JSONL output. Nó cũng sinh các insight định lượng từ metric và dữ liệu candidate.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/content/Text-Mining---RAG-on-News")

OUTPUT_CANDIDATES = [
    PROJECT_ROOT / "src" / "reranker" / "output_Rerank" / "rerank_structure_jina_top5.jsonl",
    PROJECT_ROOT / "src" / "reranker" / "output" / "rerank_structure_jina_top5.jsonl",
]
OUTPUT_PATH = next((path for path in OUTPUT_CANDIDATES if path.exists()), OUTPUT_CANDIDATES[0])

print("Project root:", PROJECT_ROOT)
print("Rerank output:", OUTPUT_PATH)
if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy output rerank. Đã kiểm tra: {OUTPUT_CANDIDATES}"
    )

In [ ]:
rows = []
with OUTPUT_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON lỗi tại dòng {line_number}") from exc

if not rows:
    raise ValueError("Output rerank không có dòng dữ liệu.")

def metric_value(metrics, name):
    """Read metric keys case-insensitively and tolerate @ notation."""
    target = name.lower()
    for key, value in (metrics or {}).items():
        if str(key).lower() == target:
            return float(value)
    return np.nan

summary_records = []
candidate_records = []

for row in rows:
    rerank_metrics = row.get("rerank_metrics", {})
    original_metrics = row.get("original_retrieval_metrics", {})
    base = {
        "qa_id": row.get("qa_id"),
        "qa_type": row.get("qa_type", "unknown"),
        "question": row.get("question", ""),
        "top_k_input": int(row.get("top_k_input", 0) or 0),
        "top_n_output": int(row.get("top_n_output", 0) or 0),
        "num_candidates": len(row.get("reranked_candidates", []) or []),
    }
    for metric in ["hit@1", "hit@5", "recall@5", "mrr@5", "ndcg@5"]:
        base[f"rerank_{metric}"] = metric_value(rerank_metrics, metric)
    for metric in ["hit@1", "hit@5", "recall@5", "mrr@10", "ndcg@10"]:
        base[f"retrieval_{metric}"] = metric_value(original_metrics, metric)
    summary_records.append(base)

    for candidate in row.get("reranked_candidates", []) or []:
        candidate_records.append({
            "qa_id": row.get("qa_id"),
            "qa_type": row.get("qa_type", "unknown"),
            "rank": candidate.get("rank"),
            "original_rank": candidate.get("original_rank"),
            "article_id": candidate.get("article_id"),
            "chunk_id": candidate.get("chunk_id"),
            "rerank_score": candidate.get("rerank_score"),
            "retrieval_score": candidate.get("retrieval_score"),
            "text_chars": len(candidate.get("text", "") or ""),
        })

summary_df = pd.DataFrame(summary_records)
candidate_df = pd.DataFrame(candidate_records)

print(f"Queries: {len(summary_df):,}")
print(f"Candidate rows: {len(candidate_df):,}")
display(summary_df.head())

In [ ]:
metric_columns = [
    "rerank_hit@1", "rerank_hit@5", "rerank_recall@5",
    "rerank_mrr@5", "rerank_ndcg@5"
]
overview = pd.DataFrame({
    "metric": metric_columns,
    "mean": [summary_df[column].mean() for column in metric_columns],
    "std": [summary_df[column].std() for column in metric_columns],
    "min": [summary_df[column].min() for column in metric_columns],
    "max": [summary_df[column].max() for column in metric_columns],
}).round(4)

coverage = pd.Series({
    "queries": len(summary_df),
    "queries_with_empty_output": int((summary_df["num_candidates"] == 0).sum()),
    "candidate_coverage": float((summary_df["num_candidates"] > 0).mean()),
    "mean_input_candidates": summary_df["top_k_input"].mean(),
    "mean_output_candidates": summary_df["num_candidates"].mean(),
    "unique_qa_types": summary_df["qa_type"].nunique(),
}).to_frame("value").round(4)

display(overview)
display(coverage)

In [ ]:
plot_df = overview.copy()
plot_df["label"] = plot_df["metric"].str.replace("rerank_", "", regex=False)

ax = sns.barplot(data=plot_df, x="label", y="mean", color=sns.color_palette()[0])
ax.set_title("Reranker performance")
ax.set_xlabel("Metric")
ax.set_ylabel("Mean score")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)
plt.tight_layout()
plt.show()

In [ ]:
if not candidate_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(
        data=candidate_df,
        x="rerank_score",
        bins=30,
        kde=True,
        ax=axes[0],
        color=sns.color_palette()[0],
    )
    axes[0].set_title("Distribution of rerank scores")
    axes[0].set_xlabel("Rerank score")

    score_by_rank = candidate_df.groupby("rank", as_index=False)["rerank_score"].mean()
    sns.lineplot(
        data=score_by_rank,
        x="rank",
        y="rerank_score",
        marker="o",
        ax=axes[1],
        color=sns.color_palette()[1],
    )
    axes[1].set_title("Mean score by final rank")
    axes[1].set_xlabel("Final rank")
    axes[1].set_ylabel("Mean rerank score")
    axes[1].set_xticks(sorted(score_by_rank["rank"].dropna().unique()))
    plt.tight_layout()
    plt.show()

    display(candidate_df["rerank_score"].describe().to_frame("rerank_score").round(4))

In [ ]:
by_type = (
    summary_df.groupby("qa_type")[["rerank_hit@1", "rerank_hit@5", "rerank_mrr@5"]]
    .mean()
    .sort_values("rerank_hit@1", ascending=False)
    .round(4)
)
display(by_type)

plot_type = by_type.reset_index().melt(
    id_vars="qa_type", var_name="metric", value_name="score"
)
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_type, x="qa_type", y="score", hue="metric")
plt.title("Reranker performance by QA type")
plt.xlabel("QA type")
plt.ylabel("Mean score")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
movement = candidate_df.dropna(subset=["rank", "original_rank"]).copy()
if not movement.empty:
    movement["rank_change"] = movement["original_rank"] - movement["rank"]
    print("Positive rank change means the candidate moved upward after reranking.")
    display(movement["rank_change"].describe().to_frame("rank_change").round(4))

    plt.figure(figsize=(10, 5))
    sns.histplot(movement["rank_change"], discrete=True, color=sns.color_palette()[2])
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Candidate rank movement after reranking")
    plt.xlabel("Original rank - final rank")
    plt.ylabel("Number of candidates")
    plt.tight_layout()
    plt.show()
else:
    print("Không có original_rank trong output; không thể phân tích rank movement.")

In [ ]:
insights = []

empty_rate = float((summary_df["num_candidates"] == 0).mean())
if empty_rate == 0:
    insights.append("Coverage tốt: mọi query đều có candidate sau rerank.")
else:
    insights.append(f"Có {empty_rate:.1%} query không có candidate; cần kiểm tra input retrieval hoặc bước tạo candidates.")

top1 = summary_df["rerank_hit@1"].mean()
top5 = summary_df["rerank_hit@5"].mean()
insights.append(f"Hit@1 trung bình là {top1:.3f}, còn Hit@5 là {top5:.3f}; khoảng cách {(top5 - top1):.3f} cho biết lợi ích của việc giữ thêm top-5 kết quả.")

if not by_type.empty:
    best_type = by_type["rerank_hit@1"].idxmax()
    worst_type = by_type["rerank_hit@1"].idxmin()
    insights.append(
        f"QA type tốt nhất theo Hit@1 là {best_type} ({by_type.loc[best_type, 'rerank_hit@1']:.3f}); "
        f"thấp nhất là {worst_type} ({by_type.loc[worst_type, 'rerank_hit@1']:.3f})."
    )

if not movement.empty:
    moved_up = int((movement["rank_change"] > 0).sum())
    moved_down = int((movement["rank_change"] < 0).sum())
    unchanged = int((movement["rank_change"] == 0).sum())
    insights.append(
        f"Rank movement: {moved_up:,} candidate tăng hạng, {moved_down:,} giảm hạng, "
        f"{unchanged:,} giữ nguyên."
    )
else:
    insights.append("Không có dữ liệu rank gốc; cần lưu original_rank để đánh giá reranker thay đổi thứ hạng thế nào.")

print("Insights")
for insight in insights:
    print("-", insight)